# Master's Degree AI Assistant using Retrieval-Augmented Generation (RAG)

## Introduction

This notebook implements a local Retrieval-Augmented Generation (RAG) system for a Master's degree academic assistant.

The goal of the project is to build a question-answering system that can answer student questions using official Master's degree documentation as its knowledge base. The system uses information from PDF documents, saved HTML pages, and official university webpages.

Instead of relying only on the internal knowledge of a Large Language Model, the assistant first retrieves relevant document chunks and then generates an answer grounded in those sources.

This project connects several topics from the course, including text mining, document parsing, vector space models, semantic embeddings, indexing and retrieval, transformers, Large Language Models, and Retrieval-Augmented Generation.

The implemented pipeline follows these steps:

1. Load PDF, HTML, TXT, and webpage sources.
2. Extract and clean raw text.
3. Split documents into overlapping chunks.
4. Convert chunks into multilingual sentence embeddings.
5. Store embeddings in a FAISS vector index.
6. Retrieve the most relevant chunks for a user question.
7. Build a grounded prompt using the retrieved context.
8. Generate an answer using different local LLMs.
9. Compare the behavior of different models.
10. Discuss limitations and possible improvements.

In [3]:
!pip uninstall -y numpy pandas scipy scikit-learn
!pip install -q --no-cache-dir --force-reinstall \
numpy==1.26.4 \
pandas==2.2.2 \
scipy==1.12.0 \
scikit-learn==1.4.2

!pip install -q --no-cache-dir \
faiss-cpu==1.8.0 \
huggingface_hub==0.34.0 \
transformers==4.44.2 \
sentence-transformers==3.0.1 \
tokenizers==0.19.1 \
pymupdf \
beautifulsoup4 \
requests==2.32.4 \
tqdm \
sentencepiece \
accelerate

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
Found existing installation: scipy 1.16.3
Uninstalling scipy-1.16.3:
  Successfully uninstalled scipy-1.16.3
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 884.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.8/37.8 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 420.8 MB/s eta

In [1]:
import os
import re
import json
import fitz
import requests
import numpy as np
import pandas as pd
import faiss
import torch

from bs4 import BeautifulSoup
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
BASE_DIR = "/content/rag_project"

PDF_DIR = os.path.join(BASE_DIR, "pdfs")
HTML_DIR = os.path.join(BASE_DIR, "html")
TXT_DIR = os.path.join(BASE_DIR, "txt")

for folder in [BASE_DIR, PDF_DIR, HTML_DIR, TXT_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Project folders created:")
print(PDF_DIR)
print(HTML_DIR)
print(TXT_DIR)

Project folders created:
/content/rag_project/pdfs
/content/rag_project/html
/content/rag_project/txt


In [3]:
from pathlib import Path

BASE_DIR = Path("rag_project")

PDF_DIR = BASE_DIR / "pdfs"
HTML_DIR = BASE_DIR / "html"
TXT_DIR = BASE_DIR / "txt"

pdf_files = list(PDF_DIR.glob("*.pdf"))
html_files = list(HTML_DIR.glob("*.html")) + list(HTML_DIR.glob("*.htm"))
txt_files = list(TXT_DIR.glob("*.txt"))

print(f"Found {len(pdf_files)} PDF files")
print(f"Found {len(html_files)} HTML files")
print(f"Found {len(txt_files)} TXT files")

Found 0 PDF files
Found 0 HTML files
Found 0 TXT files


In [5]:
print("PDF files:")
print(os.listdir(PDF_DIR))

print("\nHTML files:")
print(os.listdir(HTML_DIR))

print("\nTXT files:")
print(os.listdir(TXT_DIR))

PDF files:
['Triptico_42x21_2_compressed.pdf', 'MSTCcalendar2025-26.pdf', 'Competences.pdf', 'Acceso_y_Admision_MUTSC.pdf', 'MSTCschedule2025-26_S2.pdf', 'MSTCexams2025-26_X.pdf', 'MSTCexams2025-26_S2.pdf', 'Quality.pdf', '2016_17_TarjetaUniversitariaguiaPV.pdf', 'Declaracion_Responsable_Finalizacion_Estudios.pdf', 'presentacion_acto_ de_acogida_ 25-26.pdf', 'Tutores_TFM_MUTSC.pdf']

HTML files:
['¿Por qué no puedo acceder a mi asignatura?.html', 'Moodle – Soporte UPM.html', '¿Cómo conocer tu calificación en Moodle?.html', 'MOODLE UPM - OFICIALES 25-26: Acceso desde la App UPM Moodle | MOODLE UPM - OFICIALES 25-26.html', 'Help and documentation for students: Using Zoom | MOODLE UPM - OFFICIALS 25-26.html', '¿Cómo funcionan los cuestionarios?.html', 'Cómo utilizar el Foro.html', 'Universidad Politécnica de Madrid.html', '¿Cómo funcionan las tareas?.html']

TXT files:
[]


In [6]:
links_data = [
    {
        "url": "https://www.upm.es/Estudiantes/Estudios_Titulaciones/Estudios_Master/Programas?id=9.17&fmt=detail",
        "description": "General information about the Master's degree"
    },

    {
        "url": "https://ssr.upm.es/mutsc/",
        "description": "Landing page of the Master's program"
    },

    {
        "url": "https://ssr.upm.es/mutsc/#informacion_practica",
        "description": "Practical information for the Master's program"
    },

    {
        "url": "https://ssr.upm.es/en/",
        "description": "Department information and overview"
    },

    {
        "url": "https://www.upm.es/Estudiantes/Estudios_Titulaciones/Estudios_Master/Calendario",
        "description": "Pre-registration calendar"
    },

    {
        "url": "https://www.upm.es/Estudiantes/Estudios_Titulaciones/Estudios_Master/Admision",
        "description": "Admission requirements"
    },

    {
        "url": "https://www.upm.es/Estudiantes/Estudios_Titulaciones/Estudios_Master/Calendario?prefmt=articulo&fmt=detail&id=CON07704",
        "description": "Prices and tuition fees"
    },

    {
        "url": "https://www.upm.es/Estudiantes/Estudios_Titulaciones/Estudios_Master/Calendario?prefmt=articulo&fmt=detail&id=CON06502",
        "description": "Information about enabling university masters"
    },

    {
        "url": "https://www.upm.es/Estudiantes/Estudios_Titulaciones/Estudios_Master/Matricula",
        "description": "Registration and enrollment process"
    },

    {
        "url": "https://www.upm.es/UPM/Historia/ResenaHistorica",
        "description": "History of the university"
    },

    {
        "url": "https://www.upm.es/UPM/Centros",
        "description": "University centers and campuses"
    },

    {
        "url": "https://serviciosgate.upm.es/recursos/52/learner-guide-for-moodle-use",
        "description": "Moodle learner guide"
    }
]

print(f"Loaded {len(links_data)} web sources")

Loaded 12 web sources


## Data Extraction

In this section, we extract raw text from all available sources: PDF files, HTML files, and official university webpages.

This step is part of the text mining pipeline. The goal is to transform unstructured documents into plain text, so that they can later be cleaned, split into chunks, indexed, and searched.

In the course theory, this corresponds to the initial stages of text mining: parsing, feature extraction, and indexing.

In [7]:
def clean_text(text):
    """
    Basic text cleaning function.
    It removes excessive whitespace and normalizes line breaks.
    """
    text = re.sub(r"\s+", " ", text)
    return text.strip()

### PDF Text Extraction

PDF files often contain important official information, such as course guides, regulations, academic calendars, and administrative instructions.

We use PyMuPDF to read each PDF page by page. For each page, we store both the extracted text and useful metadata, such as the filename and page number. This metadata will later allow the assistant to cite the source of each answer.

In [8]:
def extract_text_from_pdfs(pdf_dir):
    documents = []

    for filename in os.listdir(pdf_dir):
        if filename.lower().endswith(".pdf"):
            file_path = os.path.join(pdf_dir, filename)

            try:
                pdf = fitz.open(file_path)

                for page_number, page in enumerate(pdf, start=1):
                    text = page.get_text()
                    text = clean_text(text)

                    if len(text) > 50:
                        documents.append({
                            "text": text,
                            "source": filename,
                            "source_type": "pdf",
                            "page": page_number,
                            "description": "PDF document"
                        })

            except Exception as e:
                print(f"Error reading PDF: {filename}")
                print(e)

    return documents


pdf_documents = extract_text_from_pdfs(PDF_DIR)

print(f"Extracted {len(pdf_documents)} PDF pages.")

Extracted 36 PDF pages.


### HTML File Extraction

Some academic information may be stored as saved HTML pages rather than PDFs.  
HTML pages usually contain structured content, but they also include menus, scripts, styles, and navigation elements.

We use BeautifulSoup to remove unnecessary HTML elements and keep only the meaningful textual content.

In [9]:
def extract_text_from_html_files(html_dir):
    documents = []

    for filename in os.listdir(html_dir):
        if filename.lower().endswith((".html", ".htm")):
            file_path = os.path.join(html_dir, filename)

            try:
                with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                    html = f.read()

                soup = BeautifulSoup(html, "html.parser")

                for tag in soup(["script", "style", "nav", "footer"]):
                    tag.decompose()

                text = soup.get_text(separator=" ")
                text = clean_text(text)

                if len(text) > 50:
                    documents.append({
                        "text": text,
                        "source": filename,
                        "source_type": "html",
                        "page": None,
                        "description": "Saved HTML document"
                    })

            except Exception as e:
                print(f"Error reading HTML: {filename}")
                print(e)

    return documents


html_documents = extract_text_from_html_files(HTML_DIR)

print(f"Extracted {len(html_documents)} HTML documents.")

Extracted 9 HTML documents.


### Webpage Extraction from URLs

In addition to local files, the knowledge base includes official university webpages.  
Instead of using screenshots, we directly retrieve the webpage content and extract the text.

This is similar to the crawling and indexing process discussed in the course theory: we collect webpages, extract their text, and prepare them for indexing and search.

In [10]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

def scrape_webpage(url):
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        for tag in soup(["script", "style", "nav", "footer"]):
            tag.decompose()

        text = soup.get_text(separator=" ")
        text = clean_text(text)

        return text

    except Exception as e:
        print(f"Error scraping URL: {url}")
        print(e)
        return ""


web_documents = []

for item in tqdm(links_data):
    text = scrape_webpage(item["url"])

    if len(text) > 50:
        web_documents.append({
            "text": text,
            "source": item["url"],
            "source_type": "webpage",
            "page": None,
            "description": item["description"]
        })

print(f"Extracted {len(web_documents)} webpages.")

100%|██████████| 12/12 [00:12<00:00,  1.02s/it]

Extracted 12 webpages.


In [11]:
documents = pdf_documents + html_documents + web_documents

print(f"Total documents/passages collected: {len(documents)}")

Total documents/passages collected: 57


In [12]:
documents[0]

{'text': 'MSTC MASTER IN SIGNAL THEORY AND COMMUNICATIONS UPM: ESCUELA TÉCNICA SUPERIOR DE INGENIEROS DE TELECOMUNICACIÓN Avda. Complutense, 30 - 28040 Madrid www.etsit.upm.es 91 336 73 51 mstc.ssr.upm.es MSTC Oﬀered by UPM-ETSIT-SSR, the Master of Science in Signal Theory and Communications (MSTC) is a multi-disciplinary program providing graduates the capabilities to join any national or international university for Ph.D. or high-tech company. The curriculum provides students high comprehension by case studies orientation and personalized guidance with an individual advisor. It also oﬀers balanced theoretical, mathematical capabilities, practical and laboratory skills. PROGRAM The Master degree is structured in three tracks. Two tracks correspond to classical topics representing most of national and international technological activities where our department is leader in Spain. The third track deals with new challenges in signal processing and machine learning for big data. This trac

## Text Chunking

After extracting text from PDFs, HTML files, and webpages, the next step is to split the text into smaller passages called chunks.

Chunking is necessary because:
- long documents cannot be sent entirely to the language model
- retrieval works better when the system searches smaller, focused text passages
- each chunk can be indexed and compared with the user's query

This step is related to the text preprocessing and indexing pipeline discussed in the course theory, where raw text is transformed into structured representations for search and retrieval.

In [13]:
def create_chunks(documents, chunk_size=800, chunk_overlap=150):
    chunks = []

    for doc_id, doc in enumerate(documents):
        text = doc["text"]

        start = 0
        chunk_id = 0

        while start < len(text):
            end = start + chunk_size
            chunk_text = text[start:end]

            if len(chunk_text.strip()) > 100:
                chunks.append({
                    "chunk_id": len(chunks),
                    "document_id": doc_id,
                    "text": chunk_text.strip(),
                    "source": doc["source"],
                    "source_type": doc["source_type"],
                    "page": doc["page"],
                    "description": doc["description"]
                })

            chunk_id += 1
            start += chunk_size - chunk_overlap

    return chunks


chunks = create_chunks(
    documents,
    chunk_size=800,
    chunk_overlap=150
)

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 279


In [14]:
chunks[0]

{'chunk_id': 0,
 'document_id': 0,
 'text': 'MSTC MASTER IN SIGNAL THEORY AND COMMUNICATIONS UPM: ESCUELA TÉCNICA SUPERIOR DE INGENIEROS DE TELECOMUNICACIÓN Avda. Complutense, 30 - 28040 Madrid www.etsit.upm.es 91 336 73 51 mstc.ssr.upm.es MSTC Oﬀered by UPM-ETSIT-SSR, the Master of Science in Signal Theory and Communications (MSTC) is a multi-disciplinary program providing graduates the capabilities to join any national or international university for Ph.D. or high-tech company. The curriculum provides students high comprehension by case studies orientation and personalized guidance with an individual advisor. It also oﬀers balanced theoretical, mathematical capabilities, practical and laboratory skills. PROGRAM The Master degree is structured in three tracks. Two tracks correspond to classical topics representing most of national a',
 'source': 'Triptico_42x21_2_compressed.pdf',
 'source_type': 'pdf',
 'page': 1,
 'description': 'PDF document'}

In [15]:
print("Source:", chunks[0]["source"])
print("Type:", chunks[0]["source_type"])
print("Page:", chunks[0]["page"])
print("\nText:")
print(chunks[0]["text"])

Source: Triptico_42x21_2_compressed.pdf
Type: pdf
Page: 1

Text:
MSTC MASTER IN SIGNAL THEORY AND COMMUNICATIONS UPM: ESCUELA TÉCNICA SUPERIOR DE INGENIEROS DE TELECOMUNICACIÓN Avda. Complutense, 30 - 28040 Madrid www.etsit.upm.es 91 336 73 51 mstc.ssr.upm.es MSTC Oﬀered by UPM-ETSIT-SSR, the Master of Science in Signal Theory and Communications (MSTC) is a multi-disciplinary program providing graduates the capabilities to join any national or international university for Ph.D. or high-tech company. The curriculum provides students high comprehension by case studies orientation and personalized guidance with an individual advisor. It also oﬀers balanced theoretical, mathematical capabilities, practical and laboratory skills. PROGRAM The Master degree is structured in three tracks. Two tracks correspond to classical topics representing most of national a


## Embedding Generation

In this step, each text chunk is transformed into a dense numerical vector called an embedding.

Unlike traditional Bag-of-Words or TF-IDF representations, embeddings capture semantic similarity between texts. This means that two passages can be considered similar even if they do not contain exactly the same words.

Since the dataset contains both English and Spanish sources, we use a multilingual embedding model. This allows the system to retrieve relevant Spanish passages even when the user asks a question in English, and vice versa.

In [16]:
embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedding_model = SentenceTransformer(embedding_model_name)

print("Embedding model loaded:")
print(embedding_model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Embedding model loaded:
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [17]:
chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Embeddings shape: (279, 384)


In [18]:
embeddings = embeddings.astype("float32")

faiss.normalize_L2(embeddings)

print("Embeddings normalized for cosine similarity.")

Embeddings normalized for cosine similarity.


## Vector Indexing with FAISS

After generating embeddings, we store them in a vector index.

A vector index allows fast similarity search. When the user asks a question, the question is also converted into an embedding, and FAISS retrieves the chunks with the most similar vectors.

Because the embeddings are L2-normalized, inner product similarity is equivalent to cosine similarity.

In [19]:
embedding_dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dimension)
index.add(embeddings)

print(f"FAISS index created with {index.ntotal} vectors.")

FAISS index created with 279 vectors.


In [20]:
def retrieve(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        chunk = chunks[idx].copy()
        chunk["score"] = float(score)
        results.append(chunk)

    return results

In [21]:
test_query = "What are the admission requirements for the master's degree?"

results = retrieve(test_query, top_k=5)

for i, result in enumerate(results, start=1):
    print("=" * 80)
    print(f"Result {i}")
    print("Score:", result["score"])
    print("Source:", result["source"])
    print("Type:", result["source_type"])
    print("Page:", result["page"])
    print("Description:", result["description"])
    print()
    print(result["text"][:1000])

Result 1
Score: 0.5339453816413879
Source: Acceso_y_Admision_MUTSC.pdf
Type: pdf
Page: 1
Description: PDF document

homologación de sus títulos, previa comprobación por la Universidad de que aquellos acreditan un nivel de formación equivalente a los correspondientes títulos universitarios oficiales españoles y que facultan en el país expedidor del título para el acceso a enseñanzas de postgrado. El acceso por esta vía no implicará, en ningún caso, la homologación del título previo de que esté en posesión el interesado, ni su reconocimiento a otros efectos que el de cursar las enseñanzas de Máster. PERFIL DE INGRESO PERFIL A (perfil de ingreso recomendado): Títulos con asignaturas relacionadas con el análisis y procesamiento de señal, sistemas multimedia, teoría de las comunicaciones, sobre todo a nivel físico, y los sistemas de radiofrecuencia. Por ejemplo: - Grado en Ingeniería de Tecnologías y Servici
Result 2
Score: 0.5296609997749329
Source: Triptico_42x21_2_compressed.pdf
Type: pd

## Retrieval-Augmented Answer Generation

After retrieving the most relevant chunks, the next step is to generate a natural language answer.

The language model does not answer from its internal knowledge only. Instead, it receives:
- the user's question
- the retrieved chunks from the knowledge base
- the source metadata

This is the main idea of Retrieval-Augmented Generation (RAG): first retrieve relevant information, then generate an answer grounded in that information.

In [22]:
def build_context(retrieved_chunks):
    context_parts = []

    for i, chunk in enumerate(retrieved_chunks, start=1):
        source_info = f"Source {i}: {chunk['source']}"

        if chunk["page"] is not None:
            source_info += f", page {chunk['page']}"

        source_info += f"\nDescription: {chunk['description']}"

        context_parts.append(
            source_info + "\n" + chunk["text"]
        )

    return "\n\n---\n\n".join(context_parts)

In [23]:
def build_prompt(question, retrieved_chunks):

    context = build_context(retrieved_chunks)

    prompt = f"""
You are a university academic assistant.

Read the context carefully and answer the question in English.

Summarize the relevant information clearly.
Do not copy the context verbatim.
Use only the provided context.
Mention the source document.

Context:
{context}

Question:
{question}

Answer in English:
"""

    return prompt

In [24]:
question = "What are the admission requirements for the master's degree?"

retrieved_chunks = retrieve(question, top_k=5)

prompt = build_prompt(question, retrieved_chunks)

print(prompt[:4000])


You are a university academic assistant.

Read the context carefully and answer the question in English.

Summarize the relevant information clearly.
Do not copy the context verbatim.
Use only the provided context.
Mention the source document.

Context:
Source 1: Acceso_y_Admision_MUTSC.pdf, page 1
Description: PDF document
homologación de sus títulos, previa comprobación por la Universidad de que aquellos acreditan un nivel de formación equivalente a los correspondientes títulos universitarios oficiales españoles y que facultan en el país expedidor del título para el acceso a enseñanzas de postgrado. El acceso por esta vía no implicará, en ningún caso, la homologación del título previo de que esté en posesión el interesado, ni su reconocimiento a otros efectos que el de cursar las enseñanzas de Máster. PERFIL DE INGRESO PERFIL A (perfil de ingreso recomendado): Títulos con asignaturas relacionadas con el análisis y procesamiento de señal, sistemas multimedia, teoría de las comunicaci

## Local LLM Generation with HuggingFace

Instead of using a cloud API, we can run an open-source language model locally inside Google Colab.

This allows the RAG system to work without external API costs while still generating grounded answers from the retrieved context.

In [25]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model loaded successfully:", model_name)

Model loaded successfully: google/flan-t5-large


In [26]:
def ask_rag_local(question, top_k=5):

    retrieved_chunks = retrieve(question, top_k=top_k)

    prompt = build_prompt(question, retrieved_chunks)

    answer = generate_answer(prompt)

    return {
        "question": question,
        "answer": answer,
        "retrieved_chunks": retrieved_chunks
    }

In [27]:
def generate_answer(prompt, max_new_tokens=120):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.5,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [28]:
result = ask_rag_local(
    "What are the admission requirements for the master's degree?",
    top_k=2
)

print(result["answer"])

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:615: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(


The Master degree is structured in three tracks. Two tracks correspond to classical topics representing most of national and international technological activities where our department is leader in Spain. The third track deals with new challenges in signal processing and machine learning for big data.


## Final RAG Question Answering Interface

The following function combines:
- semantic retrieval
- context construction
- grounded answer generation
- source attribution

This provides the final interface of the academic assistant.

In [29]:
def ask_and_display(question, top_k=2):

    result = ask_rag_local(question, top_k=top_k)

    print("=" * 80)
    print("QUESTION:")
    print(question)

    print("\n" + "=" * 80)
    print("ANSWER:")
    print(result["answer"])

    print("\n" + "=" * 80)
    print("SOURCES:")

    for i, chunk in enumerate(result["retrieved_chunks"], start=1):

        print(f"\nSource {i}")
        print("Document:", chunk["source"])
        print("Type:", chunk["source_type"])

        if chunk["page"] is not None:
            print("Page:", chunk["page"])

        print("Similarity score:", round(chunk["score"], 4))

        print("\nChunk preview:")
        print(chunk["text"][:400])

        print("\n" + "-" * 80)

In [30]:
ask_and_display(
    "What is the registration process?"
)

ask_and_display(
    "How can students use Moodle?"
)

ask_and_display(
    "What information is available about the university campuses?"
)

QUESTION:
What is the registration process?

ANSWER:
The registration process is based on the following:

SOURCES:

Source 1
Document: https://www.upm.es/Estudiantes/Estudios_Titulaciones/Estudios_Master/Matricula
Type: webpage
Similarity score: 0.4929

Chunk preview:
enda leer "¿Qué tengo que hacer antes de matricularme?" . Los procesos previos como la Activación de la cuenta @alumnos.upm.es no se activan hasta comenzada la matrícula. También está abierta la posibilidad de realizar la matrícula a través de la Secretaría de la Escuela responsable de la gestión del máster, dentro del plazo establecido, cuando exista algún problema para realizar la Automatrícula.

--------------------------------------------------------------------------------

Source 2
Document: Competences.pdf
Type: pdf
Page: 2
Similarity score: 0.3981

Chunk preview:
Competencias Transversales: 5 registros encontrados, mostrando todos los registros. Código Denominación Tipo CT1 Capacidad para comprender los contenidos

Improved Generation with Instruction-Following LLMs

The baseline RAG pipeline successfully retrieved relevant chunks from the knowledge base, but the final generated answers were not always satisfactory.

The main issues observed were incomplete answers, mixed Spanish and English text, weak summarization, occasional copying of raw context, and truncated final sentences.

To improve the quality of the final answers, we keep the same retrieval pipeline and replace only the generation model with stronger instruction-following LLMs.

## Loading a Stronger Instruction-Following Model

In this improved version, we use Qwen2.5-1.5B-Instruct as the local language model.

The embedding model and FAISS index remain the same.  
Only the answer generation component is changed.

In [31]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

llm_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(llm_model_name)

llm_model = AutoModelForCausalLM.from_pretrained(
    llm_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Improved LLM loaded successfully.")

Improved LLM loaded successfully.


In [32]:
def build_context_v2(retrieved_chunks, max_chars_per_chunk=700):
    context_parts = []

    for i, chunk in enumerate(retrieved_chunks, start=1):
        source_info = f"[Source {i}] {chunk['source']}"

        if chunk["page"] is not None:
            source_info += f", page {chunk['page']}"

        text = chunk["text"][:max_chars_per_chunk]

        context_parts.append(
            source_info + "\n" + text
        )

    return "\n\n".join(context_parts)

## Improved Prompt

The improved prompt explicitly asks the model to answer in English, avoid copying the context, and avoid inventing information.

This helps reduce hallucinations and improves the readability of the answer.

In [33]:
def build_prompt_v2(question, retrieved_chunks):
    context = build_context_v2(retrieved_chunks)

    prompt = f"""
You are an academic assistant for Master's degree students.

Answer in clear English.
Use only the provided context.
Do not copy the context word-for-word.
Do not invent information.
If the answer is not available in the context, say:
"The information is not available in the provided sources."

Context:
{context}

Question:
{question}

Write a complete answer in English and mention the source.
"""

    return prompt

## Improved Answer Generation

The new generation function uses a chat-style prompt format, which is more suitable for instruction-following language models.

In [34]:
def generate_answer_v2(prompt, max_new_tokens=250):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful academic assistant. Answer clearly and only using the provided context."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(llm_model.device)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

In [35]:
def ask_rag_v2(question, top_k=2):
    retrieved_chunks = retrieve(question, top_k=top_k)
    prompt = build_prompt_v2(question, retrieved_chunks)
    answer = generate_answer_v2(prompt)

    return {
        "question": question,
        "answer": answer,
        "retrieved_chunks": retrieved_chunks
    }

In [36]:
def ask_and_display_v2(question, top_k=2):
    result = ask_rag_v2(question, top_k=top_k)

    print("=" * 80)
    print("QUESTION:")
    print(question)

    print("\n" + "=" * 80)
    print("ANSWER:")
    print(result["answer"])

    print("\n" + "=" * 80)
    print("SOURCES:")

    for i, chunk in enumerate(result["retrieved_chunks"], start=1):
        print(f"\nSource {i}")
        print("Document:", chunk["source"])
        print("Type:", chunk["source_type"])

        if chunk["page"] is not None:
            print("Page:", chunk["page"])

        print("Similarity score:", round(chunk["score"], 4))

        print("\nChunk preview:")
        print(chunk["text"][:400])

        print("\n" + "-" * 80)

In [37]:
ask_and_display_v2(
    "What is the registration process?",
    top_k=2
)

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


QUESTION:
What is the registration process?

ANSWER:
The registration process for the master's program at UPM involves several steps:

1. **Preparation**: Students should prepare by understanding the requirements of the program, including any prerequisites or specific courses they need to take before applying.

2. **Application**: Students can apply online through the official website of the university. They will need to fill out an application form with personal details, educational background, and other relevant information.

3. **Matriculation**: After submitting their application, students must go through the matriculation process. This typically includes paying the required fees and completing any necessary paperwork.

4. **Activation of Account**: The activation of the account "@alumnos.upm.es" is part of this process and occurs after the matriculation has been completed.

5. **Registration**: Once activated, students can register for classes and programs within the system.

6. *

In [38]:
ask_and_display_v2(
    "How can students use Moodle?",
    top_k=2
)

QUESTION:
How can students use Moodle?

ANSWER:
To access Moodle as a student, follow these steps:

1. Download the UPM Moodle app from either Google Play Store or Apple App Store depending on your device.
2. Open the downloaded app and follow the on-screen instructions to set up your account if you haven't already done so.
3. Once logged in, navigate through the various sections within Moodle that cater to different aspects of learning such as course management, group management, assessment tools, and more.
4. Utilize features like discussion forums, quizzes, assignments, and collaborative projects to enhance your educational experience.

The information about how to use Moodle as a student is provided in Source 2: "MOODLE UPM - OFICIALES 25-26: Acceso desde la App UPM Moodle | MOODLE UPM - OFICIALES 25-26.html".

SOURCES:

Source 1
Document: Moodle – Soporte UPM.html
Type: html
Similarity score: 0.6942

Chunk preview:
Moodle – Soporte UPM Saltar al contenido principal Perfil Solicitu

In [39]:
ask_and_display_v2(
    "What are the admission requirements for the master's degree?",
    top_k=2
)

QUESTION:
What are the admission requirements for the master's degree?

ANSWER:
The admission requirements for the master's degree as described in Source 1 include:

- Homologating their previous titles through verification by the University that they meet the equivalent level of Spanish official university degrees and faculty qualifications in the country issuing the title for access to postgraduate studies.
- Not implying any prior title recognition or other purposes beyond the ability to enroll in Masters' courses.

The source does not provide specific details about additional requirements such as TOEFL/IELTS scores, letters of recommendation, or application deadlines. For more detailed information on admission requirements, please refer to the original document [Source 1].

SOURCES:

Source 1
Document: Acceso_y_Admision_MUTSC.pdf
Type: pdf
Page: 1
Similarity score: 0.5339

Chunk preview:
homologación de sus títulos, previa comprobación por la Universidad de que aquellos acreditan u

# Additional LLM Experiment: Phi-3.5-mini

In this section, we evaluate an additional instruction-tuned Large Language Model (LLM), namely Phi-3.5-mini-instruct.

The purpose of this experiment is to compare the performance of different LLM architectures within the same Retrieval-Augmented Generation (RAG) pipeline.

To avoid GPU memory issues, the previously loaded Qwen model is removed before loading Phi.

In [40]:
# Optional third model: Phi-3.5-mini-instruct
# We delete the previously loaded Qwen model first to free GPU memory.

import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

if "llm_model" in globals():
    del llm_model

if "tokenizer" in globals():
    del tokenizer

gc.collect()
torch.cuda.empty_cache()

phi_model_name = "microsoft/Phi-3.5-mini-instruct"

phi_tokenizer = AutoTokenizer.from_pretrained(
    phi_model_name,
    trust_remote_code=True
)

phi_model = AutoModelForCausalLM.from_pretrained(
    phi_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Phi model loaded successfully.")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Phi model loaded successfully.


## Phi-based Answer Generation

This function generates answers using the Phi-3.5-mini model.

The retrieved context is transformed into a structured prompt and passed to the model through a chat template. The model then generates a final response conditioned on the retrieved information.

In [41]:
def generate_answer_phi(prompt, max_new_tokens=300):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful academic assistant. Answer clearly and only using the provided context."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = phi_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = phi_tokenizer(
        text,
        return_tensors="pt"
    ).to(phi_model.device)

    outputs = phi_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=phi_tokenizer.eos_token_id
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    answer = phi_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

In [42]:
def ask_rag_phi(question, top_k=5):
    retrieved_chunks = retrieve(question, top_k=top_k)
    prompt = build_prompt_v2(question, retrieved_chunks)
    answer = generate_answer_phi(prompt)

    return {
        "question": question,
        "answer": answer,
        "sources": retrieved_chunks
    }

## Evaluation of the Phi-based RAG System

The following experiment evaluates the complete RAG pipeline using the Phi model.

Several questions related to the master's degree are tested in order to analyze:
- retrieval quality,
- grounding on the retrieved documents,
- answer clarity,
- and overall response relevance.

In [43]:
def print_sources(sources):

    for i, source in enumerate(sources, 1):

        print("-" * 80)
        print(f"Source {i}")

        if "source" in source:
            print("Document:", source["source"])

        if "type" in source:
            print("Type:", source["type"])

        if "page" in source:
            print("Page:", source["page"])

        if "score" in source:
            print("Similarity score:", round(source["score"], 4))

        print("\nChunk preview:")

        preview = source["text"][:500].replace("\n", " ")

        print(preview)
        print()

In [44]:
test_questions = [
    "What is the registration process?",
    "How can students use Moodle?",
    "What are the admission requirements for the master's degree?"
]

for q in test_questions:
    result = ask_rag_phi(q, top_k=5)
    print("=" * 80)
    print("QUESTION:")
    print(result["question"])
    print("\nANSWER:")
    print(result["answer"])
    print("\nSOURCES:")
    print_sources(result["sources"])

The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.


QUESTION:
What is the registration process?

ANSWER:
The registration process involves several steps before officially registering for your master's program at UPC Madrid. Firstly, you must ensure that all prerequisites such as account activation on their platform (@alumnos.upm.es) have been completed prior to beginning the actual coursework ([Source 1]). If there are any issues with automatic registration through the designated secretariat responsible for managing each specific master’s curriculum within its stipulated timeframe, alternative options may be considered ("Automatrícula"). Additionally, it should be noted from Source 1 that different fees apply depending upon whether one holds comunitario or extracomunitario status along with nationality considerations related to tuition costs per credit unit across various Masters programs offered by UPC Madrid. Furthermore, while this summary does not detail every aspect of the application procedure, potential questions regarding compet

# Evaluation and Conclusions

In this section, we evaluate the implemented RAG pipeline using a small set of representative questions. The goal is to analyze both retrieval quality and generation quality.

The evaluation is qualitative, because no ground-truth answer dataset is available. Therefore, the answers are manually inspected according to relevance, grounding, completeness, and hallucination risk.

## Evaluation Questions

The following questions were used to test the system:

1. What is the registration process?
2. How can students use Moodle?
3. What are the admission requirements for the master's degree?

These questions were selected because they require information from different types of sources, including official UPM webpages, Moodle support pages, and PDF documents.

In [46]:
import pandas as pd

evaluation_results = pd.DataFrame({
    "Model": [
        "google/flan-t5-large",
        "Qwen/Qwen2.5-1.5B-Instruct",
        "microsoft/Phi-3.5-mini-instruct"
    ],

    "Answer Quality": [
        "Moderate",
        "Good",
        "Good"
    ],

    "Observed Strengths": [
        "Produces cleaner and more complete answers than smaller baseline models",
        "Good balance between fluency and concise answers",
        "Produces detailed answers and follows instructions well"
    ],

    "Observed Weaknesses": [
        "Still occasionally generates incomplete or generic answers",
        "Sometimes introduces unsupported details",
        "Can combine unrelated retrieved chunks into the same answer"
    ],

    "Main Observation": [
        "Strong improvement over smaller seq2seq baselines",
        "Generated more natural and structured responses",
        "Most detailed answers, but sometimes noisier"
    ]
})

evaluation_results

,Model,Answer Quality,Observed Strengths,Observed Weaknesses,Main Observation
0,google/flan-t5-large,Moderate,Produces cleaner and more complete answers tha...,Still occasionally generates incomplete or gen...,Strong improvement over smaller seq2seq baselines
1,Qwen/Qwen2.5-1.5B-Instruct,Good,Good balance between fluency and concise answers,Sometimes introduces unsupported details,Generated more natural and structured responses
2,microsoft/Phi-3.5-mini-instruct,Good,Produces detailed answers and follows instruct...,Can combine unrelated retrieved chunks into th...,"Most detailed answers, but sometimes noisier"


## Error Analysis

The experiments showed that answer quality depends strongly on retrieval quality.

Although the system usually retrieves relevant chunks, some retrieved passages are only partially related to the query. In these cases, the LLM may combine unrelated information into the final answer.

Other observed issues include:

- occasional hallucinations,
- truncated answers,
- mixed English and Spanish context,
- and noisy retrieved chunks.

## Final Conclusions

This project successfully implemented a local Retrieval-Augmented Generation (RAG) system for Master's degree documentation.

The system combines document parsing, chunking, multilingual embeddings, FAISS vector search, and LLM-based answer generation.

The experiments showed that instruction-tuned models such as Qwen2.5 and Phi-3.5 produced more natural and informative answers than the Flan-T5 baseline.

However, the results also showed that final answer quality depends heavily on retrieval quality and context selection.

Possible future improvements include reranking methods, hybrid retrieval, improved chunking strategies, and larger LLMs.